In [2]:
import tkinter as tk
from tkinter import ttk, messagebox
import random
import heapq
import copy
import math
from collections import deque

CUTOFF = "CUTOFF"
FAILURE = "FAILURE"

class SearchNode:
    def __init__(self, floor_state, position, parent=None, birth_action=None):
        self.floor_state = floor_state
        self.position = position
        self.parent = parent
        self.birth_action = birth_action
        self.heuristic_cost = 0

class VacuumGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("Mô phỏng AI Máy Hút Bụi")
        self.root.geometry("1000x600")

        self.m = 4
        self.n = 4
        self.room = []
        self.start_x = 0
        self.start_y = 0
        self.current_x = 0
        self.current_y = 0
        self.is_animating = False
        self.animation_id = None
        self.path = []
        self.step_idx = 0

        self.setup_ui()
        self.generate_room()

    def setup_ui(self):
        # Bố cục chính
        self.root.columnconfigure(1, weight=1)
        self.root.rowconfigure(0, weight=1)

        # === BÊN TRÁI: Bảng điều khiển ===
        left_frame = tk.Frame(self.root, width=200, bg="#f0f0f0", padx=10, pady=10)
        left_frame.grid(row=0, column=0, sticky="ns")
        left_frame.grid_propagate(False)

        tk.Label(left_frame, text="KÍCH THƯỚC SÀN", bg="#f0f0f0", font=("Arial", 10, "bold")).pack(pady=(0, 5))

        row_frame = tk.Frame(left_frame, bg="#f0f0f0")
        row_frame.pack(fill="x", pady=2)
        tk.Label(row_frame, text="Số dòng (m):", bg="#f0f0f0").pack(side="left")
        self.entry_m = tk.Entry(row_frame, width=5)
        self.entry_m.insert(0, "4")
        self.entry_m.pack(side="right")

        col_frame = tk.Frame(left_frame, bg="#f0f0f0")
        col_frame.pack(fill="x", pady=2)
        tk.Label(col_frame, text="Số cột (n):", bg="#f0f0f0").pack(side="left")
        self.entry_n = tk.Entry(col_frame, width=5)
        self.entry_n.insert(0, "4")
        self.entry_n.pack(side="right")

        self.btn_generate = tk.Button(left_frame, text="Tạo sàn mới", command=self.generate_room, bg="#d9edf7")
        self.btn_generate.pack(fill="x", pady=10)

        tk.Label(left_frame, text="THUẬT TOÁN", bg="#f0f0f0", font=("Arial", 10, "bold")).pack(pady=(15, 5))
        self.algo_var = tk.StringVar(value="BFS Loại 1")

        self.cb_algo = ttk.Combobox(left_frame, textvariable=self.algo_var, state="readonly",
                                    values=["BFS Loại 1", "BFS Loại 2",
                                            "DFS Loại 1", "DFS Loại 2",
                                            "IDS Loại 1", "IDS Loại 2",
                                            "Uniform Cost Search (UCS)",
                                            "Greedy Best-First Search",
                                            "A* Search",
                                            "IDA* Search",
                                            "Simple Hill Climbing",
                                            "Steepest Ascent Hill Climbing",
                                            "Stochastic Hill Climbing",
                                            "Random Restart Hill Climbing",
                                            "Local Beam Search",
                                            "Simulated Annealing",
                                            "Sensorless Search (No Obs)",
                                            "AND/OR Search (Partial Obs)"])
        self.cb_algo.pack(fill="x", pady=5)

        self.btn_start = tk.Button(left_frame, text="Bắt đầu", command=self.start_simulation, bg="#dff0d8", font=("Arial", 10, "bold"))
        self.btn_start.pack(fill="x", pady=10)

        self.btn_stop = tk.Button(left_frame, text="Kết thúc", command=self.stop_simulation, bg="#f2dede")
        self.btn_stop.pack(fill="x", pady=5)

        # === Ở GIỮA: Sàn nhà ===
        center_frame = tk.Frame(self.root, bg="white", bd=2, relief="sunken")
        center_frame.grid(row=0, column=1, sticky="nsew", padx=10, pady=10)

        self.canvas = tk.Canvas(center_frame, bg="white")
        self.canvas.pack(fill="both", expand=True)
        self.canvas.bind("<Configure>", lambda e: self.draw_grid() if self.room else None)

        # === BÊN PHẢI: Log hệ thống ===
        right_frame = tk.Frame(self.root, width=250)
        right_frame.grid(row=0, column=2, sticky="ns", padx=10, pady=10)
        right_frame.grid_propagate(False)

        tk.Label(right_frame, text="LOG CÁC BƯỚC CHẠY", font=("Arial", 10, "bold")).pack()
        self.txt_log = tk.Text(right_frame, width=30, state="disabled", font=("Courier", 9))
        scrollbar = tk.Scrollbar(right_frame, command=self.txt_log.yview)
        self.txt_log.config(yscrollcommand=scrollbar.set)
        scrollbar.pack(side="right", fill="y")
        self.txt_log.pack(side="left", fill="both", expand=True)

        # === BÊN DƯỚI: Kết quả đường đi ===
        bottom_frame = tk.Frame(self.root, height=120, bg="#e8e8e8", bd=2, relief="groove")
        bottom_frame.grid(row=1, column=0, columnspan=3, sticky="ew")
        bottom_frame.pack_propagate(False)

        tk.Label(bottom_frame, text="KẾT QUẢ ĐƯỜNG ĐI:", font=("Arial", 10, "bold"), bg="#e8e8e8").pack(anchor="w", padx=10, pady=(5,0))
        self.lbl_result = tk.Label(bottom_frame, text="Chưa có dữ liệu.", font=("Arial", 10), bg="#e8e8e8", fg="blue", wraplength=950, justify="left")
        self.lbl_result.pack(anchor="w", padx=10, fill="x")

    def log(self, message):
        self.txt_log.config(state="normal")
        self.txt_log.insert(tk.END, message + "\n")
        self.txt_log.see(tk.END)
        self.txt_log.config(state="disabled")

    def clear_log(self):
        self.txt_log.config(state="normal")
        self.txt_log.delete(1.0, tk.END)
        self.txt_log.config(state="disabled")

    def generate_room(self):
        self.stop_simulation()
        try:
            self.m = int(self.entry_m.get())
            self.n = int(self.entry_n.get())
            if self.m <= 0 or self.n <= 0: raise ValueError
        except ValueError:
            messagebox.showerror("Lỗi nhập liệu", "Kích thước m, n phải là số nguyên dương.")
            return

        self.room = [[random.randint(0,1) for _ in range(self.n)] for _ in range(self.m)]

        self.start_x = random.randint(0, self.m - 1)
        self.start_y = random.randint(0, self.n - 1)
        self.room[self.start_x][self.start_y] = 0

        self.current_x, self.current_y = self.start_x, self.start_y

        self.clear_log()
        self.log(f"Đã tạo sàn {self.m}x{self.n}")
        self.log(f"Vị trí bắt đầu: ({self.start_x}, {self.start_y})")
        self.lbl_result.config(text="Sẵn sàng...", fg="black")

        self.draw_grid()

    def draw_grid(self):
        self.canvas.delete("all")
        if not self.room: return

        c_width = self.canvas.winfo_width()
        c_height = self.canvas.winfo_height()
        if c_width <= 1: c_width = 400
        if c_height <= 1: c_height = 400

        cell_w = c_width / self.n
        cell_h = c_height / self.m

        for i in range(self.m):
            for j in range(self.n):
                x1, y1 = j * cell_w, i * cell_h
                x2, y2 = x1 + cell_w, y1 + cell_h

                color = "#d3d3d3" if self.room[i][j] == 1 else "#ffffff"
                self.canvas.create_rectangle(x1, y1, x2, y2, fill=color, outline="black")

                if i == self.current_x and j == self.current_y:
                    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
                    r = min(cell_w, cell_h) * 0.3
                    self.canvas.create_oval(cx-r, cy-r, cx+r, cy+r, fill="#4CAF50", outline="darkgreen", width=2)
                    self.canvas.create_text(cx, cy, text="M", fill="white", font=("Arial", int(r), "bold"))

    def is_clean(self, room_state):
        for row in room_state:
            if 1 in row:
                return False
        return True

    def count_trash(self, room_state):
        return sum(row.count(1) for row in room_state)

    def get_children(self, current_room, x, y):
        children = []
        def clean(nx, ny):
            copy_room = [list(row) for row in current_room]
            copy_room[nx][ny] = 0
            return tuple(tuple(row) for row in copy_room)

        if x > 0: children.append((clean(x-1,y), x - 1, y, "UP"))
        if x < self.m - 1: children.append((clean(x+1,y), x + 1, y, "DOWN"))
        if y > 0: children.append((clean(x,y-1), x, y - 1, "LEFT"))
        if y < self.n - 1: children.append((clean(x,y+1), x, y + 1, "RIGHT"))
        return children

    def get_neighbors_pure(self, x, y):
        neighbors = []
        if x > 0: neighbors.append((x - 1, y, "UP"))
        if y > 0: neighbors.append((x, y - 1, "LEFT"))
        if x < self.m - 1: neighbors.append((x + 1, y, "DOWN"))
        if y < self.n - 1: neighbors.append((x, y + 1, "RIGHT"))
        return neighbors

    # ================= THUẬT TOÁN TÍCH HỢP MỚI =================

    def sensorless_search(self, start_room):
        """Tìm kiếm không có quan sát (Mù hoàn toàn)"""
        m, n = self.m, self.n
        start_room_tuple = tuple(tuple(row) for row in start_room)
        initial_positions = frozenset((i, j) for i in range(m) for j in range(n))

        frontier = deque([(start_room_tuple, initial_positions, [])])
        reached = set()

        def transition_belief(current_room, belief_positions, action):
            next_positions = set()
            next_room = [list(row) for row in current_room]

            for (x, y) in belief_positions:
                if action == "UP":     nx, ny = max(0, x - 1), y
                elif action == "DOWN": nx, ny = min(m - 1, x + 1), y
                elif action == "LEFT": nx, ny = x, max(0, y - 1)
                elif action == "RIGHT":nx, ny = x, min(n - 1, y + 1)
                next_positions.add((nx, ny))

            if len(next_positions) == 1:
                (only_x, only_y) = list(next_positions)[0]
                next_room[only_x][only_y] = 0

            return tuple(tuple(row) for row in next_room), frozenset(next_positions)

        while frontier:
            curr_room, curr_beliefs, path = frontier.popleft()

            if self.is_clean(curr_room):
                return path, len(reached)

            signature = (curr_room, curr_beliefs)
            if signature in reached:
                continue
            reached.add(signature)

            for action in ["UP", "DOWN", "LEFT", "RIGHT"]:
                next_room, next_beliefs = transition_belief(curr_room, curr_beliefs, action)
                child_signature = (next_room, next_beliefs)

                if child_signature not in reached:
                    frontier.append((next_room, next_beliefs, path + [action]))

        return None, len(reached)

    def partially_observable_search(self, start_room, start_x, start_y):
        """Tìm kiếm có quan sát một phần sử dụng cấu trúc AND/OR Search Tree"""
        m, n = self.m, self.n
        start_room_tuple = tuple(tuple(row) for row in start_room)
        initial_positions = frozenset((i, j) for i in range(m) for j in range(n))
        reached_states = set()

        def get_percept_local(x, y):
            u = "W" if x == 0 else "O"
            d = "W" if x == m - 1 else "O"
            l = "W" if y == 0 else "O"
            r = "W" if y == n - 1 else "O"
            return (u, d, l, r)

        def predict_belief(belief_positions, action):
            next_positions = set()
            for (x, y) in belief_positions:
                if action == "UP":     nx, ny = max(0, x - 1), y
                elif action == "DOWN": nx, ny = min(m - 1, x + 1), y
                elif action == "LEFT": nx, ny = x, max(0, y - 1)
                elif action == "RIGHT":nx, ny = x, min(n - 1, y + 1)
                next_positions.add((nx, ny))
            return frozenset(next_positions)

        def update_room_and_beliefs(room_tuple, pred_positions, percept):
            filtered_positions = frozenset((x, y) for (x, y) in pred_positions if get_percept_local(x, y) == percept)
            next_room = [list(row) for row in room_tuple]
            if len(filtered_positions) == 1:
                ox, oy = list(filtered_positions)[0]
                next_room[ox][oy] = 0
            return tuple(tuple(row) for row in next_room), filtered_positions

        def or_search(curr_room, curr_beliefs, path):
            state_signature = (curr_room, curr_beliefs)
            reached_states.add(state_signature)

            if self.is_clean(curr_room):
                return "GOAL"

            if state_signature in path:
                return None

            for action in ["UP", "DOWN", "LEFT", "RIGHT"]:
                pred_beliefs = predict_belief(curr_beliefs, action)
                possible_percepts = set(get_percept_local(x, y) for (x, y) in pred_beliefs)
                percept_branches = {}
                action_is_valid = True

                for percept in possible_percepts:
                    next_room, next_beliefs = update_room_and_beliefs(curr_room, pred_beliefs, percept)
                    result = or_search(next_room, next_beliefs, path + [state_signature])

                    if result is None:
                        action_is_valid = False
                        break
                    percept_branches[percept] = result

                if action_is_valid:
                    return (action, percept_branches)

            return None

        plan = or_search(start_room_tuple, initial_positions, [])
        if plan is None:
            return None, len(reached_states)

        # Trích xuất chuỗi hành động thực tế từ Decision Tree dựa trên vị trí khởi tạo thực tế của GUI
        flat_actions = []
        curr_node = plan
        curr_rx, curr_ry = start_x, start_y

        while curr_node != "GOAL" and curr_node is not None:
            action, branches = curr_node
            flat_actions.append(action)
            if action == "UP":     curr_rx = max(0, curr_rx - 1)
            elif action == "DOWN": curr_rx = min(m - 1, curr_rx + 1)
            elif action == "LEFT": curr_ry = max(0, curr_ry - 1)
            elif action == "RIGHT": curr_ry = min(n - 1, curr_ry + 1)

            real_percept = get_percept_local(curr_rx, curr_ry)
            if real_percept in branches:
                curr_node = branches[real_percept]
            else:
                break

        return flat_actions, len(reached_states)

    # ================= CÁC THUẬT TOÁN CŨ GỐC =================
    def bfs_1(self, start_room, x, y):
        start_room = tuple(tuple(row) for row in start_room)
        if self.is_clean(start_room): return [], 0
        frontier = deque([(start_room, x, y, [])])
        reached = set()
        while frontier:
            current_room, cx, cy, path = frontier.popleft()
            state_signature = (current_room, cx, cy)
            if self.is_clean(current_room): return path, len(reached)
            if state_signature in reached: continue
            reached.add(state_signature)
            for n_room, nx, ny, action in self.get_children(current_room, cx, cy):
                if (n_room, nx, ny) not in reached:
                    frontier.append((n_room, nx, ny, path + [action]))
        return None, len(reached)

    def bfs_2(self, start_room, x, y):
        start_room = tuple(tuple(row) for row in start_room)
        if self.is_clean(start_room): return [], 0
        frontier = deque([(start_room, x, y, [])])
        reached = set()
        while frontier:
            current_room, cx, cy, path = frontier.popleft()
            state_signature = (current_room, cx, cy)
            if state_signature in reached: continue
            reached.add(state_signature)
            for n_room, nx, ny, action in self.get_children(current_room, cx, cy):
                child_signature = (n_room, nx, ny)
                new_path = path + [action]
                if self.is_clean(n_room):
                    reached.add(child_signature)
                    return new_path, len(reached)
                if child_signature not in reached:
                    frontier.append((n_room, nx, ny, new_path))
        return None, len(reached)

    def dfs_1(self, start_room, x, y):
        start_room = tuple(tuple(row) for row in start_room)
        if self.is_clean(start_room): return [], 0
        frontier = deque([(start_room, x, y, [])])
        reached = set()
        while frontier:
            current_room, cx, cy, path = frontier.pop()
            state_signature = (current_room, cx, cy)
            if self.is_clean(current_room): return path, len(reached)
            if state_signature in reached: continue
            reached.add(state_signature)
            for n_room, nx, ny, action in self.get_children(current_room, cx, cy):
                if (n_room, nx, ny) not in reached:
                    frontier.append((n_room, nx, ny, path + [action]))
        return None, len(reached)

    def dfs_2(self, start_room, x, y):
        start_room = tuple(tuple(row) for row in start_room)
        if self.is_clean(start_room): return [], 0
        frontier = deque([(start_room, x, y, [])])
        reached = set()
        while frontier:
            current_room, cx, cy, path = frontier.pop()
            state_signature = (current_room, cx, cy)
            if state_signature in reached: continue
            reached.add(state_signature)
            for n_room, nx, ny, action in self.get_children(current_room, cx, cy):
                child_signature = (n_room, nx, ny)
                new_path = path + [action]
                if self.is_clean(n_room):
                    reached.add(child_signature)
                    return new_path, len(reached)
                if child_signature not in reached:
                    frontier.append((n_room, nx, ny, new_path))
        return None, len(reached)

    def ids_1(self, start_room, x, y):
        start_room = tuple(tuple(row) for row in start_room)
        total_pop = 0
        for depth in range(0, 100):
            result, p_count = self.dls_1(start_room, x, y, depth)
            total_pop += p_count
            if result != CUTOFF:
                if result == FAILURE: return None, total_pop
                return result, total_pop
        return None, total_pop

    def dls_1(self, start_room, x, y, limit):
        frontier = deque([(start_room, x, y, [], 0, frozenset([(start_room, x, y)]))])
        result = FAILURE
        p_count = 0
        while frontier:
            current_room, current_x, current_y, path, depth, ancestors = frontier.pop()
            p_count += 1
            if self.is_clean(current_room): return path, p_count
            if depth >= limit:
                result = CUTOFF
            else:
                for next_room, next_x, next_y, action in self.get_children(current_room, current_x, current_y):
                    child_signature = (next_room, next_x, next_y)
                    if child_signature not in ancestors:
                        new_path = path + [action]
                        new_ancestors = frozenset(ancestors | {child_signature})
                        frontier.append((next_room, next_x, next_y, new_path, depth + 1, new_ancestors))
        return result, p_count

    def ids_2(self, start_room, x, y):
        start_room = tuple(tuple(row) for row in start_room)
        total_pop = 0
        for depth in range(0, 100):
            result, p_count = self.dls_2(start_room, x, y, depth)
            total_pop += p_count
            if result != CUTOFF:
                if result == FAILURE: return None, total_pop
                return result, total_pop
        return None, total_pop

    def dls_2(self, start_room, x, y, limit):
        if self.is_clean(start_room): return [], 0
        frontier = deque([(start_room, x, y, [], 0, frozenset([(start_room, x, y)]))])
        result = FAILURE
        p_count = 0
        while frontier:
            current_room, current_x, current_y, path, depth, ancestors = frontier.pop()
            p_count += 1
            if depth >= limit:
                result = CUTOFF
            else:
                for next_room, next_x, next_y, action in self.get_children(current_room, current_x, current_y):
                    child_signature = (next_room, next_x, next_y)
                    if child_signature not in ancestors:
                        new_path = path + [action]
                        if self.is_clean(next_room): return new_path, p_count
                        new_ancestors = frozenset(ancestors | {child_signature})
                        frontier.append((next_room, next_x, next_y, new_path, depth + 1, new_ancestors))
        return result, p_count

    def ucs(self, start_room, x, y):
        start_room = tuple(tuple(row) for row in start_room)
        counter = 0
        frontier = []
        heapq.heappush(frontier, (0, counter, start_room, x, y, []))
        visited = set()
        pop_count = 0
        while frontier:
            cost, _, current_room, current_x, current_y, path = heapq.heappop(frontier)
            pop_count += 1
            state_signature = (current_room, current_x, current_y)
            if self.is_clean(current_room): return path, pop_count
            if state_signature in visited: continue
            visited.add(state_signature)
            for next_room, next_x, next_y, action in self.get_children(current_room, current_x, current_y):
                child_signature = (next_room, next_x, next_y)
                if child_signature not in visited:
                    counter += 1
                    heapq.heappush(frontier, (cost + 1, counter, next_room, next_x, next_y, path + [action]))
        return None, pop_count

    def greedy_search(self, start_room, start_x, start_y):
        start_room = tuple(tuple(row) for row in start_room)
        counter = 0
        frontier = []
        h_start = self.count_trash(start_room)
        heapq.heappush(frontier, (h_start, counter, start_room, start_x, start_y, []))
        frontier_set = {(start_room, start_x, start_y)}
        reached = set()
        pop_count = 0
        while frontier:
            h_cost, _, current_room, current_x, current_y, path = heapq.heappop(frontier)
            pop_count += 1
            state_signature = (current_room, current_x, current_y)
            if state_signature in frontier_set: frontier_set.remove(state_signature)
            if self.is_clean(current_room): return path, pop_count
            reached.add(state_signature)
            for next_room, next_x, next_y, action in self.get_children(current_room, current_x, current_y):
                child_signature = (next_room, next_x, next_y)
                if child_signature not in frontier_set and child_signature not in reached:
                    counter += 1
                    h_m = self.count_trash(next_room)
                    heapq.heappush(frontier, (h_m, counter, next_room, next_x, next_y, path + [action]))
                    frontier_set.add(child_signature)
        return None, pop_count

    def a_star_search(self, start_room, start_x, start_y):
        start_room = tuple(tuple(row) for row in start_room)
        counter = 0
        g_start = 0
        h_start = self.count_trash(start_room)
        frontier = []
        heapq.heappush(frontier, (g_start + h_start, counter, g_start, start_room, start_x, start_y, []))
        frontier_dict = {(start_room, start_x, start_y): g_start}
        reached = {}
        pop_count = 0
        while frontier:
            f_cost, _, g_cost, current_room, current_x, current_y, path = heapq.heappop(frontier)
            pop_count += 1
            state_signature = (current_room, current_x, current_y)
            if state_signature in frontier_dict and g_cost > frontier_dict[state_signature]: continue
            if state_signature in frontier_dict: del frontier_dict[state_signature]
            if self.is_clean(current_room): return path, pop_count
            reached[state_signature] = g_cost
            for next_room, next_x, next_y, action in self.get_children(current_room, current_x, current_y):
                child_signature = (next_room, next_x, next_y)
                g_new = g_cost + 1
                if child_signature in reached and g_new >= reached[child_signature]: continue
                if child_signature in reached: del reached[child_signature]
                if child_signature in frontier_dict:
                    if g_new < frontier_dict[child_signature]:
                        frontier_dict[child_signature] = g_new
                        counter += 1
                        heapq.heappush(frontier, (g_new + self.count_trash(next_room), counter, g_new, next_room, next_x, next_y, path + [action]))
                    continue
                if child_signature not in frontier_dict and child_signature not in reached:
                    frontier_dict[child_signature] = g_new
                    counter += 1
                    heapq.heappush(frontier, (g_new + self.count_trash(next_room), counter, g_new, next_room, next_x, next_y, path + [action]))
        return None, pop_count

    def ida_star_search(self, start_room, start_x, start_y):
        start_room = tuple(tuple(row) for row in start_room)
        total_pop = 0
        for depth in range(0, 100):
            result, p_count = self.dls_1(start_room, start_x, start_y, depth)
            total_pop += p_count
            if result != CUTOFF:
                if result == FAILURE: return None, total_pop
                return result, total_pop
        return None, total_pop

    def simple_hill_climbing(self, start_room, start_x, start_y):
        current_room = [list(row) for row in start_room]
        cx, cy = start_x, start_y
        path = []
        max_steps = 50
        step = 0
        while step < max_steps:
            if current_room[cx][cy] == 1: current_room[cx][cy] = 0
            if self.is_clean(current_room): break
            current_value = current_room[cx][cy]
            neighbors = self.get_neighbors_pure(cx, cy)
            found_better_move = False
            for next_x, next_y, direction in neighbors:
                if current_room[next_x][next_y] > current_value:
                    cx, cy = next_x, next_y
                    path.append(direction)
                    found_better_move = True
                    break
            if not found_better_move: break
            step += 1
        return path, step + 1

    def steepest_ascent_hill_climbing(self, start_room, start_x, start_y):
        current_room = [list(row) for row in start_room]
        cx, cy = start_x, start_y
        path = []
        max_steps = 50
        step = 0
        while step < max_steps:
            if current_room[cx][cy] == 1: current_room[cx][cy] = 0
            if self.is_clean(current_room): break
            current_value = current_room[cx][cy]
            neighbors = self.get_neighbors_pure(cx, cy)
            best_neighbors_list = []
            max_neighbor_value = -float('inf')
            for next_x, next_y, direction in neighbors:
                next_value = current_room[next_x][next_y]
                if next_value > max_neighbor_value:
                    max_neighbor_value = next_value
                    best_neighbors_list = [(next_x, next_y, direction)]
                elif next_value == max_neighbor_value:
                    best_neighbors_list.append((next_x, next_y, direction))
            best_neighbor = random.choice(best_neighbors_list) if best_neighbors_list else None
            if best_neighbor and max_neighbor_value > current_value:
                cx, cy, direction = best_neighbor
                path.append(direction)
            else:
                break
            step += 1
        return path, step + 1

    def stochastic_hill_climbing(self, start_room, start_x, start_y):
        current_room = [list(row) for row in start_room]
        cx, cy = start_x, start_y
        path = []
        max_steps = 50
        step = 0
        while step < max_steps:
            if current_room[cx][cy] == 1: current_room[cx][cy] = 0
            if self.is_clean(current_room): break
            current_value = current_room[cx][cy]
            neighbors = self.get_neighbors_pure(cx, cy)
            better_neighbors = []
            for next_x, next_y, direction in neighbors:
                if current_room[next_x][next_y] > current_value:
                    better_neighbors.append((next_x, next_y, direction))
            if len(better_neighbors) == 0:
                break
            else:
                next_x, next_y, direction = random.choice(better_neighbors)
                cx, cy = next_x, next_y
                path.append(direction)
            step += 1
        return path, step + 1

    def random_restart_hill_climbing(self, start_room, start_x, start_y):
        MAX_RESTART = 5
        room_working = [list(row) for row in start_room]
        global_path = []
        total_steps = 0

        for i in range(1, MAX_RESTART + 1):
            if i == 1:
                cx, cy = start_x, start_y
            else:
                cx = random.randint(0, self.m - 1)
                cy = random.randint(0, self.n - 1)
                global_path.append(f"TELEPORT_TO_({cx},{cy})")

            step = 0
            max_steps_per_run = 20

            while step < max_steps_per_run:
                total_steps += 1
                if room_working[cx][cy] == 1:
                    room_working[cx][cy] = 0

                if self.is_clean(room_working):
                    return global_path, total_steps

                current_value = room_working[cx][cy]
                neighbors = self.get_neighbors_pure(cx, cy)

                better_neighbors = []
                for next_x, next_y, direction in neighbors:
                    if room_working[next_x][next_y] > current_value:
                        better_neighbors.append((next_x, next_y, direction))

                if len(better_neighbors) == 0:
                    break
                else:
                    next_x, next_y, direction = random.choice(better_neighbors)
                    cx, cy = next_x, next_y
                    global_path.append(direction)

                step += 1

        return global_path, total_steps

    def local_beam_search_algo(self, start_room, start_x, start_y, beam_k=3):
        root = SearchNode(floor_state=start_room, position=(start_x, start_y), parent=None, birth_action=None)
        root.heuristic_cost = self.evaluate_beam_value(start_x, start_y, start_room)

        if self.is_clean(start_room):
            return self.generate_path_list(root, True), 0

        start_floor_tuple = tuple(tuple(row) for row in start_room)
        visited_global = {(start_floor_tuple, (start_x, start_y))}

        started_moves = self.get_neighbors_pure(start_x, start_y)
        current_nodes = []

        for next_x, next_y, move in started_moves:
            temp_floor = [list(row) for row in start_room]
            if temp_floor[next_x][next_y] == 1:
                temp_floor[next_x][next_y] = 0

            floor_tuple = tuple(tuple(row) for row in temp_floor)
            state_key = (floor_tuple, (next_x, next_y))

            if state_key not in visited_global:
                visited_global.add(state_key)
                temp_node = SearchNode(floor_state=temp_floor, position=(next_x, next_y), parent=root, birth_action=move)
                temp_node.heuristic_cost = self.evaluate_beam_value(next_x, next_y, temp_floor)

                if self.is_clean(temp_floor):
                    return self.generate_path_list(temp_node, True), 1
                current_nodes.append(temp_node)

        if not current_nodes:
            return self.generate_path_list(root, False), 1

        max_steps = 1000
        step = 1

        while step < max_steps:
            step += 1
            neighbors_states = []

            for node in current_nodes:
                neigh_possible_moves = self.get_neighbors_pure(node.position[0], node.position[1])

                for next_x, next_y, move in neigh_possible_moves:
                    neigh_floor = [list(row) for row in node.floor_state]
                    if neigh_floor[next_x][next_y] == 1:
                        neigh_floor[next_x][next_y] = 0

                    floor_tuple = tuple(tuple(row) for row in neigh_floor)
                    state_key = (floor_tuple, (next_x, next_y))

                    if state_key not in visited_global:
                        visited_global.add(state_key)
                        neigh_node = SearchNode(floor_state=neigh_floor, position=(next_x, next_y), parent=node, birth_action=move)
                        neigh_node.heuristic_cost = self.evaluate_beam_value(next_x, next_y, neigh_floor)

                        if self.is_clean(neigh_floor):
                            return self.generate_path_list(neigh_node, True), step
                        neighbors_states.append(neigh_node)

            if not neighbors_states:
                break

            neighbors_states.sort(key=lambda x: x.heuristic_cost)
            current_nodes = neighbors_states[:beam_k]

        if current_nodes:
            best_node = min(current_nodes, key=lambda x: x.heuristic_cost)
            return self.generate_path_list(best_node, False), step
        return [], step

    def evaluate_beam_value(self, target_x, target_y, current_room):
        trash_count = self.count_trash(current_room)
        min_dist = 0
        if trash_count > 0:
            distances = []
            for i in range(self.m):
                for j in range(self.n):
                    if current_room[i][j] == 1:
                        distances.append(abs(target_x - i) + abs(target_y - j))
            min_dist = min(distances) if distances else 0
        return (trash_count * 100) + min_dist

    def generate_path_list(self, node, success_boolean):
        path = []
        curr = node
        while curr and curr.birth_action:
            path.append(curr.birth_action)
            curr = curr.parent
        path.reverse()
        return path

    def simulated_annealing_algo(self, start_room, start_x, start_y):
        room_working = [list(row) for row in start_room]
        cx, cy = start_x, start_y

        T = 100.0
        Tmin = 0.1
        alpha = 0.95

        global_path = []
        total_steps = 0

        if room_working[cx][cy] == 1:
            room_working[cx][cy] = 0

        while T > Tmin:
            total_steps += 1
            if self.is_clean(room_working):
                return global_path, total_steps

            moves = self.get_neighbors_pure(cx, cy)
            next_x, next_y, direction = random.choice(moves)

            current_cost = self.count_trash(room_working)

            next_room_sim = [list(row) for row in room_working]
            if next_room_sim[next_x][next_y] == 1:
                next_room_sim[next_x][next_y] = 0
            next_cost = self.count_trash(next_room_sim)

            delta = next_cost - current_cost

            if delta < 0:
                cx, cy = next_x, next_y
                room_working[cx][cy] = 0
                global_path.append(direction)
            else:
                p = math.exp(-delta / T)
                if random.uniform(0, 1) < p:
                    cx, cy = next_x, next_y
                    room_working[cx][cy] = 0
                    global_path.append(direction)
                else:
                    global_path.append(f"STAY_AT_({cx},{cy})")

            T = alpha * T

        return global_path, total_steps

    # ================= ĐIỀU KHIỂN & MÔ PHỎNG UI =================
    def start_simulation(self):
        if self.is_animating: return

        self.current_x, self.current_y = self.start_x, self.start_y
        room_copy = [list(row) for row in self.room]

        self.draw_grid()
        self.clear_log()

        algo_name = self.algo_var.get()
        self.log(f"--- Đang chạy: {algo_name} ---")
        self.root.update()

        actions, metric_count = None, 0

        if algo_name == "BFS Loại 1": actions, metric_count = self.bfs_1(room_copy, self.start_x, self.start_y)
        elif algo_name == "BFS Loại 2": actions, metric_count = self.bfs_2(room_copy, self.start_x, self.start_y)
        elif algo_name == "DFS Loại 1": actions, metric_count = self.dfs_1(room_copy, self.start_x, self.start_y)
        elif algo_name == "DFS Loại 2": actions, metric_count = self.dfs_2(room_copy, self.start_x, self.start_y)
        elif algo_name == "IDS Loại 1": actions, metric_count = self.ids_1(room_copy, self.start_x, self.start_y)
        elif algo_name == "IDS Loại 2": actions, metric_count = self.ids_2(room_copy, self.start_x, self.start_y)
        elif algo_name == "Uniform Cost Search (UCS)": actions, metric_count = self.ucs(room_copy, self.start_x, self.start_y)
        elif algo_name == "Greedy Best-First Search": actions, metric_count = self.greedy_search(room_copy, self.start_x, self.start_y)
        elif algo_name == "A* Search": actions, metric_count = self.a_star_search(room_copy, self.start_x, self.start_y)
        elif algo_name == "IDA* Search": actions, metric_count = self.ida_star_search(room_copy, self.start_x, self.start_y)
        elif algo_name == "Simple Hill Climbing": actions, metric_count = self.simple_hill_climbing(room_copy, self.start_x, self.start_y)
        elif algo_name == "Steepest Ascent Hill Climbing": actions, metric_count = self.steepest_ascent_hill_climbing(room_copy, self.start_x, self.start_y)
        elif algo_name == "Stochastic Hill Climbing": actions, metric_count = self.stochastic_hill_climbing(room_copy, self.start_x, self.start_y)
        elif algo_name == "Random Restart Hill Climbing": actions, metric_count = self.random_restart_hill_climbing(room_copy, self.start_x, self.start_y)
        elif algo_name == "Local Beam Search": actions, metric_count = self.local_beam_search_algo(room_copy, self.start_x, self.start_y, beam_k=2)
        elif algo_name == "Simulated Annealing": actions, metric_count = self.simulated_annealing_algo(room_copy, self.start_x, self.start_y)
        # Khớp liên kết 2 giải thuật mới vào điều khiển UI
        elif algo_name == "Sensorless Search (No Obs)": actions, metric_count = self.sensorless_search(room_copy)
        elif algo_name == "AND/OR Search (Partial Obs)": actions, metric_count = self.partially_observable_search(room_copy, self.start_x, self.start_y)

        is_local_search = algo_name in ["Simple Hill Climbing", "Steepest Ascent Hill Climbing", "Stochastic Hill Climbing", "Random Restart Hill Climbing", "Local Beam Search", "Simulated Annealing"]

        if actions is None or (len(actions) == 0 and not self.is_clean(self.room)):
            self.lbl_result.config(text=f"Không tìm thấy đường đi! (Số trạng thái niềm tin đã duyệt/Node mở rộng: {metric_count})", fg="red")
            self.log("Không tìm thấy đường đi!")
        else:
            self.path = actions
            self.step_idx = 0
            self.is_animating = True

            if is_local_search:
                self.lbl_result.config(text=f"Thuật toán kết thúc tính toán! | Số vòng quét: {metric_count}\nĐang chạy mô phỏng trực quan...", fg="orange")
            elif algo_name in ["Sensorless Search (No Obs)", "AND/OR Search (Partial Obs)"]:
                self.lbl_result.config(text=f"Tổng số bước: {len(actions)} | Số trạng thái niềm tin (Belief States) đã duyệt: {metric_count}\nĐang mô phỏng...", fg="purple")
            else:
                self.lbl_result.config(text=f"Tổng số bước: {len(actions)} | Số node mở rộng: {metric_count}\nĐang mô phỏng...", fg="green")

            self.log(f"Bắt đầu mô phỏng trực quan...")
            self.animate_step()

    def animate_step(self):
        if not self.is_animating: return

        if self.step_idx < len(self.path):
            act = self.path[self.step_idx]
            self.step_idx += 1

            self.log(f"Bước {self.step_idx}: {act}")

            if "TELEPORT_TO_" in act:
                coords = act.replace("TELEPORT_TO_(", "").replace(")", "").split(",")
                self.current_x = int(coords[0])
                self.current_y = int(coords[1])
            elif "STAY_AT_" in act:
                coords = act.replace("STAY_AT_(", "").replace(")", "").split(",")
                self.current_x = int(coords[0])
                self.current_y = int(coords[1])
            else:
                if act == "UP": self.current_x -= 1
                elif act == "DOWN": self.current_x += 1
                elif act == "LEFT": self.current_y -= 1
                elif act == "RIGHT": self.current_y += 1
                self.room[self.current_x][self.current_y] = 0

            self.draw_grid()
            self.animation_id = self.root.after(500, self.animate_step)
        else:
            self.is_animating = False
            self.log("--- HOÀN THÀNH MÔ PHỎNG BƯỚC ĐI ---")

            if self.is_clean(self.room):
                self.lbl_result.config(text=f"Thành công: Phòng đã sạch bóng hoàn toàn! (Tổng bước: {len(self.path)})", fg="green")
            else:
                self.lbl_result.config(text=f"Dừng lại: Thuật toán đã hết bước/bị kẹt cục bộ nhưng phòng CHƯA SẠCH hoàn toàn (Còn {self.count_trash(self.room)} ô bẩn)!", fg="red")

    def stop_simulation(self):
        self.is_animating = False
        if self.animation_id:
            self.root.after_cancel(self.animation_id)
            self.animation_id = None
        self.log("Đã dừng mô phỏng.")

if __name__ == "__main__":
    root = tk.Tk()
    app = VacuumGUI(root)
    root.mainloop()

Exception in Tkinter callback
Traceback (most recent call last):
  File "C:\Users\ASUS\AppData\Local\Programs\Python\Python313\Lib\tkinter\__init__.py", line 2068, in __call__
    return self.func(*args)
           ~~~~~~~~~^^^^^^^
  File "C:\Users\ASUS\AppData\Local\Programs\Python\Python313\Lib\tkinter\__init__.py", line 862, in callit
    func(*args)
    ~~~~^^^^^^^
  File "C:\Users\ASUS\AppData\Local\Temp\ipykernel_18392\3227742214.py", line 916, in animate_step
    self.room[self.current_x][self.current_y] = 0
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^
IndexError: list assignment index out of range
